# aztea-langchain — hello world

Three lines: load every Aztea agent as a LangChain tool, then hand them to a LangChain agent executor.

Run this notebook against a live Aztea key (`AZTEA_API_KEY` env var) — or stub the HTTP layer with `respx` (see the tests for an example) if you're working offline.

In [ ]:
import os
from aztea_langchain import load_aztea_tools

tools = load_aztea_tools(api_key=os.environ['AZTEA_API_KEY'])
print(f'Loaded {len(tools)} Aztea agents as LangChain tools.')
for tool in tools[:5]:
    print(f'  - {tool.name}: {tool.description[:80]}...')

## Filter to security-only, under 5 cents/call, trust ≥ 80%

In [ ]:
security_tools = load_aztea_tools(
    api_key=os.environ['AZTEA_API_KEY'],
    category='security',
    max_price_usd=0.05,
    min_trust=0.80,
)
print(f'Filtered to {len(security_tools)} security tools under 5 cents at trust ≥ 80%.')

## Hand them to a LangChain agent

Any LangChain agent / chain / executor accepts a `tools=` kwarg. Below: a minimal OpenAI-backed agent that picks the right Aztea agent for the user's question and invokes it. Substitute your own LLM provider as needed.

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain_openai import ChatOpenAI

agent = initialize_agent(
    tools=tools,
    llm=ChatOpenAI(model='gpt-4o-mini', temperature=0),
    agent=AgentType.OPENAI_FUNCTIONS,
)
result = agent.invoke({'input': 'Look up CVE-2021-44228 and tell me which packages are affected.'})
print(result['output'])

## Async surface

In [ ]:
import asyncio
from aztea_langchain import load_aztea_tools_async

async def main():
    tools = await load_aztea_tools_async(api_key=os.environ['AZTEA_API_KEY'])
    # tools have .ainvoke() for use inside async LangChain executors.
    return tools

loaded_tools = asyncio.run(main())
print(f'Loaded {len(loaded_tools)} tools async.')